# Tumulus LiDAR Detector

Detects burial mounds (tumuli) in Romania's 0.5 m airborne LiDAR, in your browser. No install.

## To start: open the Runtime menu, click Run all, and wait a minute or two.

It sets up, shows a map, and scans a default area automatically. Then just click anywhere in the blue on the map to scan that spot. The blue is where the model can scan (0.5 m LiDAR, ANCPI LAKI III; mostly Oltenia and SW Romania).

<img src="https://raw.githubusercontent.com/ObuObuHub/tumulus-lidar-detector/main/assets/coverage_preview.png" width="460">

Form is not proof: only fieldwork confirms a tumulus. Repo: https://github.com/ObuObuHub/tumulus-lidar-detector

In [ ]:
#@title Setup (run once) { display-mode: "form" }
import os
if not os.path.isdir('tumulus-lidar-detector') and os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    !git clone -q https://github.com/ObuObuHub/tumulus-lidar-detector.git
if os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    %cd tumulus-lidar-detector
!pip install -q pyproj ipyleaflet
print("Ready. See the map below.")

In [ ]:
#@title Map - a default area is scanned to start; click anywhere in the blue to scan that spot { display-mode: "form" }
import os, sys, json, subprocess
import pandas as pd
from IPython.display import display, Image, HTML, clear_output
from ipywidgets import Output, HTML as WHTML
try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass
from ipyleaflet import Map, GeoJSON, Marker, basemaps

area_km = 1  #@param {type:"slider", min:1, max:5, step:1}

_cov = json.load(open('assets/laki3_coverage.geojson'))

def _inside(lon, lat):
    for f in _cov['features']:
        ring = f['geometry']['coordinates'][0]
        c = False; n = len(ring); j = n - 1
        for i in range(n):
            xi, yi = ring[i]; xj, yj = ring[j]
            if ((yi > lat) != (yj > lat)) and (lon < (xj - xi) * (lat - yi) / (yj - yi) + xi):
                c = not c
            j = i
        if c:
            return True
    return False

_out = Output()
_busy = {'on': False}

def _scan(lat, lon):
    if _busy['on']:
        return
    _busy['on'] = True
    try:
        with _out:
            clear_output(wait=True)
            if not _inside(lon, lat):
                print("That point is outside the blue 0.5 m coverage. Click inside the blue area.")
                return
            print("Scanning  lat %.4f, lon %.4f  (%d km box) ... about a minute." % (lat, lon, area_km))
            for _f in ('/tmp/zone_dets.csv', 'review/zone_view.jpg', 'review/zone_board.jpg', 'detected_mounds.csv'):
                if os.path.exists(_f):
                    os.remove(_f)
            subprocess.run([sys.executable, 'tools/scan_zone.py', str(lon), str(lat), str(area_km)], check=False)
            clear_output(wait=True)
            if not os.path.exists('review/zone_view.jpg'):
                print("No LiDAR tiles at this point. Click inside the blue area."); return
            n = 0
            if os.path.exists('/tmp/zone_dets.csv'):
                det = pd.read_csv('/tmp/zone_dets.csv'); kept = det[det['keep'] == 1]; n = len(kept)
            print(("Found %d probable mound(s)." % n) if n else
                  "No mounds found here. This is normal for most points: the model is selective.")
            display(Image('review/zone_view.jpg'))
            if os.path.exists('review/zone_board.jpg'):
                display(Image('review/zone_board.jpg'))
            if n:
                t = kept[['lon', 'lat', 'score', 'coh', 'pgate']].sort_values('score', ascending=False).reset_index(drop=True)
                t.insert(0, 'id', range(1, len(t) + 1))
                t.to_csv('detected_mounds.csv', index=False)
                shown = t.copy()
                shown['map'] = shown.apply(lambda r: '<a href="https://www.google.com/maps?q=%s,%s" target="_blank">view</a>' % (r.lat, r.lon), axis=1)
                display(HTML(shown.to_html(escape=False, index=False)))
    finally:
        _busy['on'] = False

_m = Map(center=(44.3, 23.6), zoom=8, scroll_wheel_zoom=True, basemap=basemaps.OpenStreetMap.Mapnik)
_m.add(GeoJSON(data=_cov, style={'color': '#1d4ed8', 'fillColor': '#2563eb', 'fillOpacity': 0.30, 'weight': 1}))
_mk = {'m': Marker(location=(44.043, 23.522))}
_m.add(_mk['m'])

def _on_click(**kw):
    if kw.get('type') == 'click':
        lat, lon = kw['coordinates']
        try:
            _m.remove(_mk['m'])
        except Exception:
            pass
        _mk['m'] = Marker(location=(lat, lon)); _m.add(_mk['m'])
        _scan(round(lat, 5), round(lon, 5))
_m.on_interaction(_on_click)

display(WHTML("<b>Click anywhere in the blue area to scan that spot</b> (about a minute). The default area below was scanned to start you off."), _m, _out)
_scan(44.043, 23.522)   # default scan so Run all shows a result